# Champion model v7.2 — SHAP-reduced retraining

This notebook retrains the v7.2 binary high-severity Random Forest using the exact 20 features selected by the completed SHAP refinement in `bottleneck_diagnosis_v7_2.ipynb`.

It saves the classifier, ordered feature list, and deployment threshold into `src/realtime/`, where the realtime pipeline loads them. Run from the repository root or `src/approach3/`.


## 1. Imports and reproducible settings


In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
HIGH_THRESHOLD_NS = 1_500_000
DECISION_THRESHOLD = 0.3701  # selected on inner validation in the v7.2 SHAP refinement


## 2. Load training data


In [ ]:
cwd = Path.cwd().resolve()
if (cwd / 'perf_metrics1.csv').exists():
    DATA_DIR = cwd
elif (cwd / 'src' / 'approach3' / 'perf_metrics1.csv').exists():
    DATA_DIR = cwd / 'src' / 'approach3'
else:
    raise FileNotFoundError('Could not locate src/approach3/perf_metrics1.csv from the current directory')

CSV_PARTS = [DATA_DIR / 'perf_metrics1.csv', DATA_DIR / 'perf_metrics2.csv']
missing_parts = [str(path) for path in CSV_PARTS if not path.is_file()]
if missing_parts:
    raise FileNotFoundError('Required training CSV part(s) are missing: ' + ', '.join(missing_parts))

df_raw = pd.concat([pd.read_csv(path) for path in CSV_PARTS], ignore_index=True)
print(f'Loaded {len(df_raw):,} rows from {len(CSV_PARTS)} CSV files')


## 3. Clean rows and reproduce v7.2 feature engineering


In [ ]:
required_raw_columns = {
    'pid', 'ctx_switches', 'latency_count', 'avg_stall_ns', 'cpu_migrations',
    'involuntary_switches', 'voluntary_switches', 'total_runtime_ns',
    'minor_faults', 'major_faults', 'mutex_contentions', 'rwsem_read_contentions',
    'rwsem_write_contentions', 'avg_mutex_wait_ns', 'syscall_count',
    'kfree_count', 'total_alloc_bytes', 'read_bytes', 'write_bytes',
    'max_syscall_latency_ns', 'avg_syscall_latency_ns',
}
missing = sorted(required_raw_columns - set(df_raw.columns))
if missing:
    raise KeyError('Training data is missing required raw columns: ' + ', '.join(missing))

df = df_raw.copy()
before = len(df)
df = df[(df['pid'] > 0) & (df['ctx_switches'] > 0) & (df['latency_count'] > 0)].copy()
print(f'Cleaned: {before:,} -> {len(df):,} rows')

df['involuntary_ratio'] = df['involuntary_switches'] / df['ctx_switches'].clip(lower=1)
df['voluntary_ratio'] = df['voluntary_switches'] / df['ctx_switches'].clip(lower=1)
df['runtime_per_switch'] = df['total_runtime_ns'] / df['ctx_switches'].clip(lower=1)
df['read_write_ratio'] = df['read_bytes'] / (df['write_bytes'] + 1)
df['alloc_pressure'] = df['total_alloc_bytes'] / df['ctx_switches'].clip(lower=1)
df['lock_pressure'] = df['mutex_contentions'] + df['rwsem_read_contentions'] + df['rwsem_write_contentions']
df['log_total_runtime_ns'] = np.log1p(df['total_runtime_ns'])
df['log_avg_syscall_latency_ns'] = np.log1p(df['avg_syscall_latency_ns'])
df['log_max_syscall_latency_ns'] = np.log1p(df['max_syscall_latency_ns'])


## 4. Freeze the post-SHAP feature order and binary labels


In [ ]:
FEATURE_COLS = ['runtime_per_switch', 'cpu_migrations', 'involuntary_ratio', 'ctx_switches', 'voluntary_switches', 'voluntary_ratio', 'log_total_runtime_ns', 'minor_faults', 'involuntary_switches', 'mutex_contentions', 'rwsem_read_contentions', 'avg_mutex_wait_ns', 'syscall_count', 'lock_pressure', 'kfree_count', 'alloc_pressure', 'read_write_ratio', 'log_max_syscall_latency_ns', 'log_avg_syscall_latency_ns', 'major_faults']
assert len(FEATURE_COLS) == 20 and len(set(FEATURE_COLS)) == 20
missing = [column for column in FEATURE_COLS if column not in df.columns]
if missing:
    raise KeyError('Engineered training data is missing SHAP-selected features: ' + ', '.join(missing))

X = df.loc[:, FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0).astype('float32')
y = (df['avg_stall_ns'] >= HIGH_THRESHOLD_NS).astype('int8')
if y.nunique() != 2:
    raise ValueError(f'Expected both binary classes; found {sorted(y.unique().tolist())}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Features: {len(FEATURE_COLS)}; train={len(X_train):,}; test={len(X_test):,}')
print('SHAP feature order:', FEATURE_COLS)


## 5. Retrain the selected tuned Random Forest

These parameters match the deployed full-feature champion used for the SHAP refinement. The model is refit on the frozen training split using only the selected 20 columns.


In [ ]:
champion_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    max_features='sqrt',
    min_samples_leaf=1,
    class_weight=None,
    n_jobs=4,
    random_state=RANDOM_STATE,
)
champion_model.fit(X_train, y_train)
champion_proba = champion_model.predict_proba(X_test)[:, 1]
champion_pred = (champion_proba >= DECISION_THRESHOLD).astype('int8')

champion_result = {
    'accuracy': accuracy_score(y_test, champion_pred),
    'f1_high': f1_score(y_test, champion_pred, zero_division=0),
    'precision_high': precision_score(y_test, champion_pred, zero_division=0),
    'recall_high': recall_score(y_test, champion_pred, zero_division=0),
    'pr_auc_high': average_precision_score(y_test, champion_proba),
    'roc_auc_high': roc_auc_score(y_test, champion_proba),
}
print(pd.Series(champion_result).round(4))
print(classification_report(y_test, champion_pred, target_names=['Not-High', 'High'], zero_division=0))


## 6. Save deployment artifacts


In [ ]:
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / 'src' / 'realtime').is_dir() else cwd.parent.parent
REALTIME_DIR = REPO_ROOT / 'src' / 'realtime'
REALTIME_DIR.mkdir(parents=True, exist_ok=True)

model_path = REALTIME_DIR / 'champion_model_v7_shap.pkl'
feature_path = REALTIME_DIR / 'feature_cols_v7_shap.json'
threshold_path = REALTIME_DIR / 'label_thresholds_v7_shap.json'
metrics_path = REALTIME_DIR / 'champion_metrics_v7_shap.csv'

joblib.dump(champion_model, model_path)
feature_path.write_text(json.dumps(FEATURE_COLS, indent=2) + '\n', encoding='utf-8')
threshold_path.write_text(json.dumps({
    'high_threshold_ns': HIGH_THRESHOLD_NS,
    'decision_probability_threshold': DECISION_THRESHOLD,
}, indent=2) + '\n', encoding='utf-8')
pd.DataFrame([champion_result]).to_csv(metrics_path, index=False)
print('Saved classifier:', model_path)
print('Saved ordered features:', feature_path)
print('Saved threshold metadata:', threshold_path)


## Realtime use

`src/realtime/config.py` defaults to the model, feature list, and threshold files saved above. The inference path selects columns in the saved SHAP order and applies the saved probability threshold. If input CSVs or the training split change, rerun the SHAP refinement and update this exact feature list and threshold together.
